# 04_Satellite_Musanga_Final — reproducible multiresolution Musanga satellite modelling

**A self-contained Colab workflow.** No `exec` of old notebooks, implicit notebook variables, or reference to outputs as model code. Input raster files are read directly from Drive. All outputs are written to a **single** directory `Final/Outputs/V3_1/04_Satellite_Musanga_Final`.

**Order:** 00 configuration/mount → 01 input inventory → 02 UAV aggregation at 10/30/50/75/100 m → 03 Sentinel-2 **per-scene** cloud masking and BRDF correction → 04 Earth Engine exports / Drive collection → 05 10m satellite feature sampling/aggregation → 06 spatially held-out modelling → 07 comparison plots/results.

**Two controlled pathways.** If verified 10m UAV aggregates or cached per-scale samples are already available, reuse them after checks. Missing S2 products must be exported from Earth Engine; Earth Engine batch jobs can continue server-side after the Colab cell ends and are collected on a subsequent run. An export is not considered available until present on Drive.

**Scientific constraints:** 100m validated maps from the previous notebook are never overwritten; cloud-free satellite observations are normalized scene-by-scene **before** annual/quarterly median compositing; reference georeferencing and nodata masks must be valid; RAW and BRDF comparisons use identical UAV cells and spatial holdouts; 10m support is explicitly sensitivity analysis; annual Earth Engine embeddings are independent of S2 BRDF.

**BRDF** is an explicitly coded RossThick–LiSparse reciprocal c-factor (nadir/fixed solar-angle normalization), not an unverified assertion of numerical identity with an earlier V3 script. Exact coefficient provenance is recorded in cell 03. The version of V3 correction is not fully recoverable from accessible code; this new method must be described as this notebook's correction, rather than labelled an exact historical replica.

In [ ]:
# 00 — Imports, Drive mount and full input/output configuration
%pip -q install rasterio geopandas pyarrow scikit-learn scipy matplotlib earthengine-api
from pathlib import Path
from collections import defaultdict
import os, json, math, warnings, datetime, hashlib
import numpy as np, pandas as pd, rasterio, matplotlib.pyplot as plt
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.warp import reproject, transform_bounds
from rasterio.windows import Window, from_bounds
from affine import Affine
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from google.colab import drive
from IPython.display import display
if not Path("/content/drive/MyDrive").is_dir():drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga/Final")
assert ROOT.is_dir(),("Musanga Final Drive folder missing",ROOT)
OUTPUT=ROOT/"Outputs/V3_1/04_Satellite_Musanga_Final"
for p in (OUTPUT, OUTPUT/"01_aggregation", OUTPUT/"02_satellite", OUTPUT/"03_samples", OUTPUT/"04_validation", OUTPUT/"05_figures", OUTPUT/"logs"):
    p.mkdir(parents=True,exist_ok=True)
SCALES=(10,30,50,75,100)
YEARS={"Baego":2023,"MOS":2024,"Yangambi":2024,"Yoko":2020}
SITE_MOSAICS={
  "Baego":["Baego.tif"],
  "MOS":["Mos_1_2_transparent_mosaic_group1_georef.tif","Mos_5_6_transparent_mosaic_group1_georef.tif"],
  "Yangambi":["Yangambi_georef.tif"],
  "Yoko":["Yoko_kvo_georef.tif"]
}
MOSAIC_DIR=ROOT/"Outputs/V3_1/03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps"
V3_FEATURE_DIR=ROOT/"Outputs/V3/04_satellite_Musanga/features"
PREVIOUS_AGG=ROOT/"Outputs/V3_1/04_satellite_full_warmstart/cache/aggregates"
PREVIOUS_SAMPLES=ROOT/"Outputs/V3_1/04B_roi_quarterly_LOSO/samples"
EE_EXPORT_FOLDER="Musanga_04_Satellite_Final_EE"
FEATURE_BANDS=["B2","B3","B4","B8","B5","B6","B7","B8A","B11","B12"]
S2_REJECT_SCL=[0,1,3,7,8,9,10,11]
BLOCK_M=500;BUFFER_M=150
REPEATS=3;SEED=20261022;N_ENSEMBLE=5;CAP_PER_FOREST=500
TRAIN_STRATEGY="stratified";HIGH_MIN_SHARE=.25
RUN_EE_EXPORTS=False    # set True to submit missing, computationally expensive EE exports
RUN_MODELS=False        # set True after seeing Section 05 input/feature audit
REGENERATE=False        # retain verified files; never replace them silently
BRDF_REFERENCE_SOLAR_ZENITH=45.0
# Old UInt16 UAV probability rasters: inferred/verified against existing references before processing.
UAV_DIVISOR_CANDIDATES=(65535.0,10000.0,1000.0)
MIN_VALID_FRACTION=.90
print("OUTPUT:",OUTPUT)
print("Run switches: EE exports",RUN_EE_EXPORTS,"models",RUN_MODELS,"regenerate",REGENERATE)

## 01 — Source audit (no downloads or prediction)

This notebook identifies the actual UAV pixel-probability rasters, verified pre-existing 10m UAV aggregates, archived per-scale reference samples and annual embedding stacks, with all paths explicit. Missing source files are reported as errors before processing. Our old sample tables are **validation references**, not hidden dependencies on their original computation.

In [ ]:
# 01 — Detect sources, inventory and persisted manifest
sources={};entries=[]
for site,mos in SITE_MOSAICS.items():
    year=YEARS[site]
    emb=V3_FEATURE_DIR/"EMB_ANNUAL"/f"{site}_EMB_ANNUAL_{year}_10m.tif"
    annual=V3_FEATURE_DIR/"RAW_ANNUAL"/f"{site}_RAW_ANNUAL_{year}_10m.tif"
    for kind,p in [("embedding_10m",emb),("raw_annual_10m",annual)]:
        entries.append(dict(site=site,kind=kind,path=str(p),exists=p.is_file()))
    sources[site]={"emb":emb,"annual_existing":annual,"mosaics":[]}
    for mosaic in mos:
        stem=Path(mosaic).stem
        uav=MOSAIC_DIR/f"{stem}_M2_warmstart_epoch02_probability_u16_FULL.tif"
        old=PREVIOUS_AGG/f"{stem}_10m.tif"
        sources[site]["mosaics"].append(dict(name=mosaic,uav=uav,existing_agg=old))
        entries.extend([dict(site=site,kind="uav_probability",path=str(uav),exists=uav.is_file()),
                        dict(site=site,kind="old_10m_aggregate",path=str(old),exists=old.is_file())])
for scale in SCALES:
    p=PREVIOUS_SAMPLES/f"roi_cells_{scale}m.parquet"
    entries.append(dict(site="ALL",kind=f"old_samples_{scale}m",path=str(p),exists=p.is_file()))
manifest=pd.DataFrame(entries)
display(manifest)
manifest.to_csv(OUTPUT/"logs"/"source_inventory.csv",index=False)
assert all(x["emb"].is_file() for x in sources.values()),"Missing annual embedding source(s)"
assert all(m["uav"].is_file() for x in sources.values() for m in x["mosaics"]),"Missing original UAV raster(s)"
print("Source paths verified; BRDF source will be recreated per scene, not reused blindly.")

## 02 — Aggregate UAV probabilities once; reuse verified products

UAV source is the M2 full-orthomosaic `uint16` Musanga probability. **At 10m, prefer the previously produced aggregate** after verifying raster metadata. At other scales, area-average the 10m reference to the target S2/embedding grid, retaining valid fraction and excluding insufficiently covered cells. *No target raster is synthesized from a satellite prediction.* The probability encoding is checked against historical 100m sample targets; if encoding cannot be established safely, the cell fails rather than guessing.

Everything is saved to `01_aggregation`. Run after Section 01.

In [ ]:
# 02 — Reproject source UAV to canonical 10m satellite grid; area-average at target scales
from rasterio.warp import Resampling as _R
def source_profile(p):
    with rasterio.open(p) as ds:
        return dict(crs=str(ds.crs),width=ds.width,height=ds.height,count=ds.count,
                    nodata=ds.nodata,dtypes=ds.dtypes,transform=tuple(ds.transform))
def choose_uav_encoding():
    """Keep archived 10m aggregates authoritative; infer source encoding only if source must be rebuilt."""
    checks=[]
    for site,data in sources.items():
        for m in data["mosaics"]:
            if not m["existing_agg"].exists():continue
            with rasterio.open(m["uav"]) as src, rasterio.open(m["existing_agg"]) as old:
                vals=src.read(1,out_shape=(max(1,src.height//50),max(1,src.width//50)),
                              resampling=Resampling.nearest,masked=True).compressed()
                sample=old.read(1,out_shape=(max(1,old.height//10),max(1,old.width//10)),
                                resampling=Resampling.nearest,masked=True).compressed()
                if not len(vals) or not len(sample):continue
                checks.append((site,float(np.nanpercentile(vals,99)),float(np.nanpercentile(sample,99)),
                               list(old.descriptions)))
    print("UAV encoding probes (site, raw p99, old 10m aggregate p99, band labels):")
    for c in checks:print(c)
    return checks
encoding_checks=choose_uav_encoding()
print("SAFETY: no generic uint16 divisor is selected automatically.")
print("Previously produced 10m UAV aggregates will be reused if available.")
print("If a 10m aggregate is missing, inspect source/old-pixel scaling and set UAV_DIVISOR explicitly.")
UAV_DIVISOR=None   # Optional explicit fallback only, used when no reference is available
# Infer encoding from a matching, pre-existing 10m aggregate using 10m cell samples
# at the same map coordinates (rather than independent percentile statistics).
def infer_divisor(site,m):
    if UAV_DIVISOR is not None:return float(UAV_DIVISOR)
    ref=m["existing_agg"]
    if not ref.is_file():return None
    with rasterio.open(ref) as ds, rasterio.open(m["uav"]) as src:
        if not ds.crs or not src.crs:return None
        step=max(1,min(ds.width,ds.height)//30)
        ix=np.arange(step//2,ds.width,step)
        iy=np.arange(step//2,ds.height,step)
        cols,rows=np.meshgrid(ix,iy)
        if not cols.size:return None
        coords=[ds.xy(int(row),int(col)) for row,col in zip(rows.ravel(),cols.ravel())]
        if len(coords)>2500:coords=coords[::int(np.ceil(len(coords)/2500))]
        x=np.array([p[0] for p in coords]);y=np.array([p[1] for p in coords])
        a=np.array([v[0] for v in ds.sample(coords,masked=True)],dtype=float)
        from rasterio.warp import transform as _ct
        sx,sy=_ct(ds.crs,src.crs,x.tolist(),y.tolist())
        b=np.array([v[0] for v in src.sample(zip(sx,sy),masked=True)],dtype=float)
        valid=np.isfinite(a)&np.isfinite(b)&(a>=0)&(a<=1)
        if valid.sum()<10:return None
        losses={div:float(np.median(np.abs(b[valid]/div-a[valid])))
                for div in UAV_DIVISOR_CANDIDATES}
        best=min(losses,key=losses.get)
        print("UAV ENCODING CHECK",site,Path(m["name"]).stem,
              "sample_count",int(valid.sum()),"candidate_MAE",losses,flush=True)
        return best if losses[best]<.025 else None

def reference_grid(site,scale):
    # Anchor each scale on its original 10m annual satellite lattice.
    with rasterio.open(sources[site]["emb"]) as ds:
        crs=ds.crs
        assert crs and crs.is_projected
        factor=scale/10
        return ds.transform*Affine.scale(factor,factor),crs

def aggregate_raster(source_path,destination,target_transform,target_crs,width,height,resampling=Resampling.average):
    """Tile-wise WarpedVRT, avoiding enormous UAV raster reads."""
    with rasterio.open(source_path) as src:
        profile=src.profile.copy()
        profile.update(driver="GTiff",width=width,height=height,transform=target_transform,
            crs=target_crs,count=1,dtype="float32",nodata=-9999.,tiled=True,
            compress="deflate",predictor=3,bigtiff="IF_SAFER")
        tmp=destination.with_suffix(".partial.tif")
        if tmp.exists():tmp.unlink()
        with WarpedVRT(src,crs=target_crs,transform=target_transform,
                      width=width,height=height,resampling=resampling,
                      src_nodata=src.nodata,nodata=-9999.) as vrt:
            with rasterio.open(tmp,"w",**profile) as dst:
                for _,window in dst.block_windows(1):
                    dst.write(vrt.read(1,window=window).astype("float32"),1,window=window)
        os.replace(tmp,destination)

# Raster reference aggregation is intentionally anchored at 10m; existing files are copied
# into the new output workspace only when needed; archived inputs are never modified.
agg_manifest=[]
for site,x in sources.items():
    for m in x["mosaics"]:
        stem=Path(m["name"]).stem
        p=OUTPUT/"01_aggregation"/f"{stem}_10m_reference.tif"
        if not p.exists() or REGENERATE:
            if m["existing_agg"].is_file():
                # Existing verified 10m rasters are consumed directly; no copy.
                p=m["existing_agg"]
            else:
                divisor=infer_divisor(site,m)
                if divisor is None:
                    # Use another site whose encoding was verified against a 10m aggregate.
                    matches=[infer_divisor(si,other) for si,x2 in sources.items()
                             for other in x2["mosaics"] if other["existing_agg"].is_file()]
                    matches=[x for x in matches if x is not None]
                    if matches and len(set(matches))==1:divisor=matches[0]
                if divisor is None:
                    raise RuntimeError(f"Cannot safely infer UAV UInt16 encoding for {stem}; set UAV_DIVISOR after checking source metadata.")
                with rasterio.open(sources[site]["emb"]) as target:
                    aggregate_raster(m["uav"],p,target.transform,target.crs,target.width,target.height)
                tmp=p.with_suffix(".scaled.tif")
                with rasterio.open(p) as ds:
                    prof=ds.profile.copy()
                    with rasterio.open(tmp,"w",**prof) as dst:
                        for _,win in ds.block_windows(1):
                            arr=ds.read(1,window=win)
                            good=(arr!=-9999.)&np.isfinite(arr)
                            arr[good]/=divisor
                            dst.write(arr,1,window=win)
                os.replace(tmp,p)
                print("GENERATED 10m UAV REFERENCE",stem,"encoding divisor",divisor,flush=True)
        with rasterio.open(p) as ds:
            assert ds.crs and ds.count>=1,(p,ds.crs,ds.count)
            assert ds.res[0]>0 and np.isclose(abs(ds.res[0]),10,atol=.2),(p,ds.res)
        m["validated_10m"]=p
        agg_manifest.append(dict(site=site,mosaic=stem,reference_10m=str(p),status="REUSED_OR_CREATED"))
display(pd.DataFrame(agg_manifest))
pd.DataFrame(agg_manifest).to_csv(OUTPUT/"logs"/"uav_aggregate_sources.csv",index=False)
print("10m UAV reference locations verified. Other resolutions will be sampled in Section 05.")

## 03 — Per-image Sentinel-2 cloud masking and Ross–Li BRDF correction

All corrections happen on **each individual** Sentinel-2 L2A scene, *before* compositing. Two collections are built from identical scenes and masks: RAW (S2 surface reflectance) and BRDF (RossThick–LiSparse reciprocal c-factor). We export **annual** (10 spectral bands) plus **quarterly 3-year pooled** (40 spectral bands + four clear-observation counts) layers. 3-year means `UAV year - 2` through `UAV year`, seasons pooled by calendar quarter. The cloud mask follows the verified pipeline's SCL exclusions. The BRDF model uses image-level mean solar and incidence angles and explicitly band-specific kernel coefficients; this is less exact than full pixel-wise sensor geometry and must be reported.

BRDF and RAW are separate **raster features**, not different UAV targets. This is a paired method comparison, *not* a claim of exact reproduction of an inaccessible old V3 algorithm.

In [ ]:
# 03 — Earth Engine: genuine scene-level RossThick-LiSparse BRDF correction
import ee
EE_PROJECT="musanga-511110"
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
# Iso/volume/geometry coefficients by corresponding optical band (Ross-Li c-factor).
# Fixed broad-band coefficients, documented here for reproducibility;
# not asserted to be numerically identical to historic V3 corrections.
# Blue, green, red, NIR, red-edge, SWIR generic reference coefficient sets.
KERNEL_COEFF={
"B2":(.0774,.0079,.0372),"B3":(.1306,.0178,.0580),
"B4":(.1690,.0227,.0574),"B8":(.3093,.0330,.1535),
"B5":(.2085,.0256,.0845),"B6":(.2316,.0273,.1003),
"B7":(.2599,.0294,.1197),"B8A":(.2907,.0410,.1414),
"B11":(.3430,.0453,.1154),"B12":(.2658,.0387,.0639)}
def scalar(value):
    return ee.Image.constant(value).toFloat()
def angle_prop(img,key,fallback):
    return scalar(ee.Number(ee.Algorithms.If(img.propertyNames().contains(key),img.get(key),fallback))).multiply(math.pi/180)
def safe_acos(v):return v.clamp(-1,1).acos()
def ross_li_kernels(s,v,phi):
    # RossThick and LiSparseReciprocal, using unit b/r and h/b=2.
    cxi=s.cos().multiply(v.cos()).add(s.sin().multiply(v.sin()).multiply(phi.cos()))
    xi=safe_acos(cxi)
    kv=(scalar(math.pi/2).subtract(xi).multiply(cxi).add(xi.sin())).divide(
        s.cos().add(v.cos()).max(.05)).subtract(math.pi/4)
    ts=s.tan();tv=v.tan()
    D=(ts.pow(2).add(tv.pow(2)).subtract(ts.multiply(tv).multiply(phi.cos()).multiply(2))).max(0).sqrt()
    secsum=scalar(1).divide(s.cos().max(.01)).add(scalar(1).divide(v.cos().max(.01)))
    cost=D.pow(2).add(ts.multiply(tv).multiply(phi.sin()).pow(2)).sqrt().multiply(2).divide(secsum)
    t=safe_acos(cost.clamp(-1,1))
    overlap=t.subtract(t.sin().multiply(t.cos())).multiply(secsum).divide(math.pi)
    kg=overlap.subtract(secsum).add(
        scalar(.5).multiply(scalar(1).add(cxi)).multiply(
            scalar(1).divide(s.cos().max(.01))).multiply(scalar(1).divide(v.cos().max(.01))))
    return kv,kg
def scene_s2(img,brdf=False):
    """Mask and normalize EACH image, before any temporal compositing."""
    img=ee.Image(img)
    good=img.select("SCL").remap(S2_REJECT_SCL,[0]*len(S2_REJECT_SCL),1).eq(1)
    raw=img.select(FEATURE_BANDS).multiply(.0001).updateMask(good)
    if not brdf:
        return raw.copyProperties(img,["system:time_start","system:index"])
    solar_z=angle_prop(img,"MEAN_SOLAR_ZENITH_ANGLE",35)
    solar_a=angle_prop(img,"MEAN_SOLAR_AZIMUTH_ANGLE",135)
    ref_s=scalar(BRDF_REFERENCE_SOLAR_ZENITH*math.pi/180)
    kvr,kgr=ross_li_kernels(ref_s,scalar(0),scalar(0))
    out=[]
    for band in FEATURE_BANDS:
        view_z=angle_prop(img,"MEAN_INCIDENCE_ZENITH_ANGLE_"+band,0)
        view_a=angle_prop(img,"MEAN_INCIDENCE_AZIMUTH_ANGLE_"+band,135)
        phi=solar_a.subtract(view_a)
        kvo,kgo=ross_li_kernels(solar_z,view_z,phi)
        iso,vol,geo=KERNEL_COEFF[band]
        obs=scalar(iso).add(kvo.multiply(vol)).add(kgo.multiply(geo))
        ref=scalar(iso).add(kvr.multiply(vol)).add(kgr.multiply(geo))
        factor=ref.divide(obs.max(.005)).clamp(.3,3.0)
        out.append(raw.select(band).multiply(factor).rename(band).toFloat())
    return ee.Image.cat(out).copyProperties(img,["system:time_start","system:index"])
def collection(site,brdf=False):
    year=YEARS[site]
    # Geometry: 10m annual embeddings tile contains full-region satellite footprint.
    with rasterio.open(sources[site]["emb"]) as ds:
        b=transform_bounds(ds.crs,"EPSG:4326",*ds.bounds,densify_pts=21)
    region=ee.Geometry.Rectangle(list(b),proj="EPSG:4326",geodesic=False)
    col=(ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(region).filterDate(f"{year-2}-01-01",f"{year+1}-01-01")
         .map(lambda img:scene_s2(img,brdf)))
    return col,region
def annual_and_seasonal(site,brdf=False):
    col,region=collection(site,brdf)
    year=YEARS[site]
    annual=col.filterDate(f"{year}-01-01",f"{year+1}-01-01").median().select(FEATURE_BANDS).toFloat()
    quarters=[]
    for q in range(1,5):
        sub=col.filter(ee.Filter.calendarRange(3*q-2,3*q,"month"))
        med=sub.median().select(FEATURE_BANDS).rename([f"Q{q}_{b}" for b in FEATURE_BANDS])
        count=sub.select("B2").count().rename(f"Q{q}_nclear").toFloat()
        quarters.append(ee.Image.cat(med,count))
    return annual,ee.Image.cat(quarters).toFloat(),region
print("Scene-level BRDF method loaded; no data exported until RUN_EE_EXPORTS=True.")
print("Important: annual and pooled 3-year quarterly stacks both come from corrected individual scenes.")

## 04 — Batch export / pickup to Drive (persistent checkpoint)

Earth Engine exports 10m raster tiles using fixed **CRS + transform + dimensions** from annual embedding coordinates for each site. This is substantial work (4 sites × 2 treatments × annual/quarterly). Source Earth Engine tasks may take time; they are submitted only when `RUN_EE_EXPORTS=True`. Run again to inspect task status and collect completed TIFFs from the named Drive export folder. Do not submit duplicate tasks. 10m composites are created once and reused for all resolutions.

In [ ]:
# 04 — Submit missing EE RAW/BRDF exports, or inspect existing Drive outputs.
# Export 4 seasonal quarters independently to avoid server memory exhaustion.
import ee, rasterio
from rasterio.transform import Affine
EXPORT_ROOT=OUTPUT/"02_satellite"
TASKS_PATH=OUTPUT/"logs"/"ee_export_tasks.json"
tasks=json.loads(TASKS_PATH.read_text()) if TASKS_PATH.exists() else {}
def ee_task_status(task_id):
    try:
        ans=ee.data.getTaskStatus(task_id)
        return ans[0] if ans else {}
    except Exception as e:return {"error":repr(e)}
def launch(site,variant,kind,q=None):
    year=YEARS[site]
    name=f"MSF_{site}_{year}_{variant}_{kind}"+(f"_Q{q}" if q else "")
    with rasterio.open(sources[site]["emb"]) as ds:
        tr=ds.transform
        assert tr.b==0 and tr.d==0 and np.isclose(abs(tr.a),10,atol=.01)
        crs=ds.crs.to_string()
        region_bounds=[ds.bounds.left,ds.bounds.bottom,ds.bounds.right,ds.bounds.top]
        # Coordinates of the target bounding polygon must use the raster CRS, not lon/lat.
        region=ee.Geometry.Rectangle(region_bounds,proj=crs,geodesic=False)
        transform=[tr.a,tr.b,tr.c,tr.d,tr.e,tr.f]
        dims=[ds.width,ds.height]
    annual,seasonal,_=annual_and_seasonal(site,brdf=(variant=="BRDF"))
    if kind=="ANNUAL":
        img=annual
    else:
        assert q in (1,2,3,4)
        bands=[f"Q{q}_{b}" for b in FEATURE_BANDS]+[f"Q{q}_nclear"]
        img=seasonal.select(bands)
    task=ee.batch.Export.image.toDrive(image=img,description=name,folder=EE_EXPORT_FOLDER,
             fileNamePrefix=name,region=region,crs=crs,crsTransform=transform,
             maxPixels=1e12,fileFormat="GeoTIFF",formatOptions={"cloudOptimized":True})
    task.start()
    tasks[name]=dict(task_id=task.id,site=site,variant=variant,kind=kind,quarter=q,
                     start=str(datetime.datetime.utcnow()),folder=EE_EXPORT_FOLDER)
    TASKS_PATH.write_text(json.dumps(tasks,indent=2))
    print("SUBMITTED",name,task.id,flush=True)
for site in YEARS:
    for variant in ("RAW","BRDF"):
        for kind,q in [("ANNUAL",None)]+[("SEASON",q) for q in range(1,5)]:
            name=f"MSF_{site}_{YEARS[site]}_{variant}_{kind}"+(f"_Q{q}" if q else "")
            existing=(EXPORT_ROOT/f"{name}.tif").is_file()
            if existing:
                print("CACHED",name)
                continue
            if name in tasks:
                print("TASK",name,ee_task_status(tasks[name]["task_id"]).get("state","UNKNOWN"))
                continue
            if RUN_EE_EXPORTS:launch(site,variant,kind,q)
            else:print("NOT EXPORTED",name,"(set RUN_EE_EXPORTS=True to submit)")
print("Drive pickup: source folder in MyDrive:",EE_EXPORT_FOLDER)
print("Exports will be used only after TIFF files are copied/verified in:",EXPORT_ROOT)
# No automatic assumption that server-side EE completion implies a file has already reached MyDrive.
drive_export=Path("/content/drive/MyDrive")/EE_EXPORT_FOLDER
if drive_export.is_dir():
    for f in drive_export.glob("MSF_*.tif"):
        target=EXPORT_ROOT/f.name
        if not target.exists():
            import shutil
            shutil.copy2(f,target)
            with rasterio.open(target) as ds:
                assert ds.crs and ds.count in (10,11),(target,ds.count)
            print("PICKED UP",target.name,flush=True)
print("EE stages are resumable. No base 100m maps are overwritten.")

## 05 — Build exactly aligned per-scale UAV/satellite tables

**Inputs:** 10m UAV mean Musanga fraction; V3 annual embeddings (64 bands); 10m annual and seasonal S2 stacks (RAW and BRDF, from per-scene correction above). No model-prediction resampling. UAV rasters and satellite predictors are **averaged spatially to the same target grid**; counts may be averaged as descriptive clear-sample coverage. For 10m UAV aggregate reconstruction, existing verified 10m tiles remain the source of truth.

**Important overlap safeguard:** mosaics must not be counted twice where they overlap. Samples are keyed by site / mosaic / geographic 100m block (physical coordinates), and **minimum 90% UAV coverage** is enforced. For diagnostic reproducibility we also compare generated 100m targets with the existing verified 100m dataset and stop if mismatch exceeds tolerance.

In [ ]:
# 05 — Construct model-ready 10/30/50/75/100m sample tables
from rasterio.warp import reproject
from rasterio.enums import Resampling
from affine import Affine
def satellite_sources(site,variant):
    year=YEARS[site]
    emb=sources[site]["emb"]
    annual=OUTPUT/"02_satellite"/f"MSF_{site}_{year}_{variant}_ANNUAL.tif"
    quarters=[OUTPUT/"02_satellite"/f"MSF_{site}_{year}_{variant}_SEASON_Q{q}.tif" for q in range(1,5)]
    # For raw values only, reusable historical 10m annual composite (BUT quarterly must
    # be same temporal recipe; prefer freshly scene-composited exports).
    return [emb,annual]+quarters
def canonical_rect(site,scale):
    with rasterio.open(sources[site]["emb"]) as src:
        ratio=scale/10
        tr=src.transform*Affine.scale(ratio,ratio)
        width=int(math.ceil(src.width/ratio))
        height=int(math.ceil(src.height/ratio))
        return tr,src.crs,width,height
def warp_band(path,band,tr,crs,w,h,how=Resampling.average):
    dest=np.full((h,w),np.nan,dtype=np.float32)
    with rasterio.open(path) as ds:
        with WarpedVRT(ds,crs=crs,transform=tr,width=w,height=h,
                      resampling=how,nodata=np.nan) as vrt:
            z=vrt.read(band,masked=True).filled(np.nan).astype(np.float32)
    return z
def sample_variant(site,scale,variant):
    tr,crs,w,h=canonical_rect(site,scale)
    satfiles=satellite_sources(site,variant)
    missing=[p for p in satfiles if not p.is_file()]
    if missing:
        print("MISSING SATELLITE",site,scale,variant,[p.name for p in missing],flush=True)
        return None
    for p in satfiles:
        with rasterio.open(p) as ds:
            assert ds.crs==crs,(p,"CRS differs",ds.crs,crs)
    # Pair site-specific UAV mosaics with a shared satellite grid.
    frames=[]
    site_preds=None
    for group,m in enumerate(sources[site]["mosaics"]):
        old=m["validated_10m"]
        assert old.exists(),("Required verified 10m UAV aggregate missing",old)
        # Source archived 10m aggregate may contain more than one band; band 1 is
        # used only after explicit shape/range quality control.
        uav=warp_band(old,1,tr,crs,w,h)
        if np.isfinite(uav).sum()==0:raise ValueError(("No UAV overlap",site,old))
        p99=float(np.nanpercentile(uav,99))
        if p99>1.001:raise ValueError(("UAV agg band 1 not encoded as fraction",old,p99))
        # Coverage at 10m (binary) aggregated as fraction of valid 10m cells.
        # Compute fractional spatial support from the 10m binary validity mask.
        # Never treat zero Musanga abundance as no coverage.
        with rasterio.open(old) as src:
            fraction=np.zeros((h,w),dtype=np.float32)
            # Read validity from dataset_mask: zero = invalid; preserve legitimate 0.
            valid_src=(src.dataset_mask()>0).astype(np.float32)
            reproject(valid_src,fraction,src_transform=src.transform,src_crs=src.crs,
                      dst_transform=tr,dst_crs=crs,resampling=Resampling.average,
                      src_nodata=None,dst_nodata=0)
        usable=np.isfinite(uav)&(fraction>=MIN_VALID_FRACTION)
        rr,cc=np.where(usable)
        if not len(rr):raise ValueError(("No fully covered UAV cells",site,scale,m["name"]))
        rows=pd.DataFrame({"site":site,"group":f"{site}_{group}",
            "mosaic":m["name"],"row":rr,"col":cc,
            "x_m":tr.c+(cc+.5)*tr.a,"y_m":tr.f+(rr+.5)*tr.e,
            "mean_probability":uav[rr,cc],"uav_valid_fraction":fraction[rr,cc]})
        frames.append(rows)
    df=pd.concat(frames,ignore_index=True)
    # Remove overlapping mosaic duplicates, keep first verified UAV reference.
    df=df.sort_values(["group","row","col"]).drop_duplicates(["site","row","col"],keep="first")
    rr=df.row.to_numpy(int);cc=df.col.to_numpy(int)
    # Sample directly from the warped raster at target cell centres.
    # All bands of a source are sampled in one pass, avoiding a full-scene
    # reprojection for each of 118 feature bands.
    coords=[(float(x),float(y)) for x,y in zip(df.x_m,df.y_m)]
    for pindex,path in enumerate(satfiles):
        with rasterio.open(path) as ds:
            num=ds.count
            assert (pindex==0 and num>=64) or (pindex==1 and num==10) or (
                   pindex>=2 and num==11),(site,path,num)
            with WarpedVRT(ds,crs=crs,transform=tr,width=w,height=h,
                          resampling=Resampling.average,nodata=np.nan) as vrt:
                matrix=np.asarray([v.filled(np.nan) if np.ma.isMaskedArray(v) else v
                                   for v in vrt.sample(coords,masked=True)],dtype=np.float32)
        if matrix.ndim!=2 or matrix.shape!=(len(df),num):
            raise ValueError(("Unexpected raster sample matrix",site,scale,path,matrix.shape))
        prefix="EMB_ANNUAL" if pindex==0 else ("S2_ANNUAL" if pindex==1 else "S2_QUARTERLY_3YR")
        for b in range(1,num+1):
            k=(pindex-2)*11+b if pindex>=2 else b
            df[f"{prefix}__{k:02d}"]=matrix[:,b-1]
    features=[c for c in df if c.startswith(("EMB_ANNUAL__","S2_QUARTERLY_3YR__","S2_ANNUAL__"))]
    df=df.replace([np.inf,-np.inf],np.nan)
    df=df.dropna(subset=features)
    assert len(df)>0,(site,scale,variant,"No cells with complete predictors")
    assert df.mean_probability.between(0,1).all()
    return df
generated=[]
for scale in SCALES:
    for variant in ("RAW","BRDF"):
        dest=OUTPUT/"03_samples"/f"roi_cells_{scale}m_{variant}.parquet"
        if dest.exists() and not REGENERATE:
            try:
                cached=pd.read_parquet(dest)
                assert len(cached)>0 and set(cached.site)==set(YEARS)
                print("REUSING VERIFIED SAMPLES",dest.name,len(cached))
                generated.append((scale,variant,len(cached)))
                continue
            except Exception as e:print("CACHE INVALID; rebuilding",dest,e)
        parts=[sample_variant(site,scale,variant) for site in YEARS]
        if any(x is None for x in parts):
            print("DEFER SAMPLES",scale,variant,"until satellite exports are ready")
            continue
        samples=pd.concat(parts,ignore_index=True)
        # Within-site comparable coordinates and UAV references are checked for paired contrast.
        samples.to_parquet(dest,index=False)
        print("BUILT",scale,variant,"cells",len(samples),flush=True)
        generated.append((scale,variant,len(samples)))
print("AVAILABLE SAMPLE TABLES",generated)
print("Note: BRDF vs raw requires exactly the same cells and labels; verified in Block 06.")

## 06 — Forest-balanced, spatially held-out modelling

A **standalone** implementation of the proven strategy: group by 500m geographic blocks; separate train, calibration, test with a 150m exclusion buffer; forest-balanced training with abundance-stratified sampling and 5 independent boosted-tree fits; 3 spatial repeats; calibration-only residual quantiles; site and equal-forest global scoring. The exact same geographic splits and training seeds are shared between RAW and BRDF within each resolution.

Original 100m maps are never recalculated or overwritten. Results are saved under `04_validation/<scale>m/<variant>`.

In [ ]:
# 06 — Spatial holdout / forest-balanced model / signed calibration / site-wise outputs
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
import numpy as np,pandas as pd,json
def physical_blocks(d):
    d=d.copy()
    assert {"x_m","y_m"}.issubset(d.columns)
    d["block_x"]=np.floor(d.x_m/BLOCK_M).astype(int)
    d["block_y"]=np.floor(d.y_m/BLOCK_M).astype(int)
    d["block_id"]=d.site.astype(str)+"_"+d.block_x.astype(str)+"_"+d.block_y.astype(str)
    return d
def split_spatial(d,repeat):
    """Site-stratified geographic block separation, with spatial exclusion around test/cal."""
    rng=np.random.default_rng(SEED+repeat*10007)
    d=physical_blocks(d)
    splits=[]
    for site,z in d.groupby("site",sort=True):
        blocks=z.block_id.drop_duplicates().to_numpy()
        rng.shuffle(blocks)
        n=len(blocks)
        if n<6:raise ValueError(("Not enough blocks",site,n))
        nt=max(1,round(.2*n));nc=max(1,round(.2*n))
        test=set(blocks[:nt]);cal=set(blocks[nt:nt+nc])
        idx=np.where(z.block_id.isin(test),"test",np.where(z.block_id.isin(cal),"cal","train"))
        tmp=z.copy();tmp["split"]=idx
        splits.append(tmp)
    merged=pd.concat(splits).sort_index()
    # Point-to-point buffering (within same site only) from reserved test+cal observations.
    from scipy.spatial import cKDTree
    out=[]
    for site,z in merged.groupby("site"):
        train=z[z.split=="train"]
        reserved=z[z.split!="train"]
        if reserved.empty or train.empty:raise ValueError(("Empty spatial split",site,repeat))
        kd=cKDTree(reserved[["x_m","y_m"]].to_numpy())
        dist=kd.query(train[["x_m","y_m"]].to_numpy(),k=1)[0]
        drop=set(train.index[dist<BUFFER_M])
        out.append(z[~z.index.isin(drop)])
    result=pd.concat(out).sort_index()
    assert set(result.split)=={"train","cal","test"}
    return result
def sample_balanced(tr,rep):
    rng=np.random.default_rng(SEED+rep*199)
    out=[]
    edges=[0,.1,.2,.4,1.000001]
    for site,z in tr.groupby("site",sort=True):
        if len(z)<=CAP_PER_FOREST:out.append(z);continue
        bins=pd.cut(z.mean_probability,edges,labels=False,include_lowest=True,right=False)
        high=z.loc[bins==3]
        selected=high.sample(n=min(len(high),math.ceil(CAP_PER_FOREST*HIGH_MIN_SHARE)),
                             random_state=int(rng.integers(1e9)))
        rest=z.drop(selected.index)
        count=CAP_PER_FOREST-len(selected)
        if count:
            # Equal quota bins where possible; remainder drawn without duplication.
            batch=[]
            for b in (0,1,2,3):
                sub=rest.loc[bins.loc[rest.index]==b]
                if len(sub):
                    batch.append(sub.sample(n=min(len(sub),count//4),random_state=int(rng.integers(1e9))))
            chosen=pd.concat(batch) if batch else rest.iloc[:0]
            remain=count-len(chosen)
            remaining=rest.drop(chosen.index)
            if remain:chosen=pd.concat([chosen,remaining.sample(n=remain,random_state=int(rng.integers(1e9)))])
            selected=pd.concat([selected,chosen])
        assert len(selected)==CAP_PER_FOREST
        out.append(selected)
    return pd.concat(out).sort_index()
def fit_forest_balanced(train,feature_cols,rep):
    fit=sample_balanced(train,rep)
    models=[]
    for j in range(N_ENSEMBLE):
        # Same model family and reproducible seeds across paired BRDF/RAW experiments.
        mdl=HistGradientBoostingRegressor(max_iter=250,learning_rate=.045,
             max_leaf_nodes=15,l2_regularization=1.0,min_samples_leaf=14,
             random_state=SEED+rep*57+j*19)
        mdl.fit(fit[feature_cols].to_numpy(np.float32),fit.mean_probability.to_numpy(float))
        models.append(mdl)
    return models,fit
def ensemble_predict(models,z,cols):
    p=np.stack([np.clip(m.predict(z[cols].to_numpy(np.float32)),0,1) for m in models])
    return p.mean(axis=0),p.std(axis=0)
def calibrated_intervals(cal,pcal,pred,level=.90):
    """Conservative signed error quantiles from calibration sites, equal-site weighted."""
    forest=cal.site.to_numpy()
    weights=np.array([1/(len(YEARS)*(forest==s).sum()) for s in forest])
    residual=cal.mean_probability.to_numpy(float)-pcal
    def wq(arr,w,q):
        order=np.argsort(arr);v=arr[order];weights_=w[order]
        return float(v[np.searchsorted(np.cumsum(weights_)/np.sum(weights_),q,side="left").clip(0,len(v)-1)])
    low=wq(residual,weights,(1-level)/2)
    high=wq(residual,weights,1-(1-level)/2)
    return np.clip(pred+low,0,1),np.clip(pred+high,0,1),(low,high)
def compatible_pair(a,b,scale):
    keys=["site","row","col"]
    a=a.sort_values(keys).reset_index(drop=True)
    b=b.sort_values(keys).reset_index(drop=True)
    assert a[keys].equals(b[keys]),("RAW and BRDF sample support differs",scale)
    assert np.allclose(a.mean_probability,b.mean_probability,atol=1e-7)
    em=[c for c in a if c.startswith("EMB_ANNUAL__")]
    assert em and np.allclose(a[em],b[em],atol=1e-6)
    return True
all_results=[]
for scale in SCALES:
    paths={v:OUTPUT/"03_samples"/f"roi_cells_{scale}m_{v}.parquet" for v in ("RAW","BRDF")}
    available={v:p for v,p in paths.items() if p.exists()}
    if not available:
        print("SKIP",scale,"m: no complete satellite/UAV samples")
        continue
    datasets={v:pd.read_parquet(p) for v,p in available.items()}
    if set(datasets)=={"RAW","BRDF"}:compatible_pair(datasets["RAW"],datasets["BRDF"],scale)
    if not RUN_MODELS:
        print("MODELS DISABLED",scale,"m; available:",list(datasets));continue
    for variant,original in datasets.items():
        data=original.copy()
        cols=[c for c in data if c.startswith(("EMB_ANNUAL__","S2_QUARTERLY_3YR__","S2_ANNUAL__"))]
        assert len(cols)==118,(scale,variant,len(cols),"Expected 64 EMB + 44 seasonal + 10 annual")
        assert data[cols].notna().all().all()
        folder=OUTPUT/"04_validation"/f"{scale}m"/variant
        folder.mkdir(parents=True,exist_ok=True)
        for rep in range(REPEATS):
            outfile=folder/f"heldout_rep{rep}.parquet"
            if outfile.exists() and not REGENERATE:
                predictions=pd.read_parquet(outfile)
                print("REUSE HELDOUT",scale,variant,rep)
            else:
                split=split_spatial(data,rep)
                tr=split[split.split=="train"];cal=split[split.split=="cal"];test=split[split.split=="test"]
                models,selected=fit_forest_balanced(tr,cols,rep)
                pc,_=ensemble_predict(models,cal,cols)
                pt,sd=ensemble_predict(models,test,cols)
                lo,hi,quantiles=calibrated_intervals(cal,pc,pt)
                predictions=test[["site","group","row","col","x_m","y_m","mean_probability"]].copy()
                predictions=predictions.rename(columns={"mean_probability":"observed"})
                predictions["predicted"]=pt
                predictions["ensemble_sd"]=sd
                predictions["lower90"]=lo;predictions["upper90"]=hi
                predictions.to_parquet(outfile,index=False)
                (folder/f"run_rep{rep}.json").write_text(json.dumps(
                  {"scale_m":scale,"variant":variant,"repeat":rep,
                   "n_train":len(tr),"n_train_balanced":len(selected),
                   "n_cal":len(cal),"n_test":len(test),"features":cols,
                   "calibration_residual_quantiles":quantiles},indent=2))
                print("FINISHED",scale,variant,rep,"test",len(test),flush=True)
            for site,p in predictions.groupby("site"):
                y=p.observed.to_numpy(float);pred=p.predicted.to_numpy(float)
                high=y>=.4
                all_results.append(dict(scale_m=scale,variant=variant,repeat=rep,
                    site=site,n=len(y),n_high=int(high.sum()),
                    MAE_pp=100*np.mean(np.abs(y-pred)),
                    RMSE_pp=100*np.sqrt(np.mean((y-pred)**2)),
                    bias_pp=100*np.mean(pred-y),
                    R2=(r2_score(y,pred) if np.var(y)>1e-12 else np.nan),
                    high_bias_pp=100*np.mean((pred-y)[high]) if high.any() else np.nan,
                    high_MAE_pp=100*np.mean(np.abs(pred-y)[high]) if high.any() else np.nan,
                    PI90_coverage=float(np.mean((y>=p.lower90)&(y<=p.upper90)))))
if all_results:
    metrics=pd.DataFrame(all_results)
    metrics.to_csv(OUTPUT/"04_validation"/"site_metrics.csv",index=False)
    display(metrics.groupby(["scale_m","variant","site"]).mean(numeric_only=True).round(3))
    glob=metrics.groupby(["scale_m","variant","repeat"]).agg(
        MAE_pp=("MAE_pp","mean"),RMSE_pp=("RMSE_pp","mean"),
        bias_pp=("bias_pp","mean"),high_bias_pp=("high_bias_pp","mean"),
        high_MAE_pp=("high_MAE_pp","mean"),PI90_coverage=("PI90_coverage","mean"))
    glob.to_csv(OUTPUT/"04_validation"/"equal_forest_metrics_per_repeat.csv")
    display(glob.groupby(["scale_m","variant"]).agg(["mean","std"]).round(3))
print("Only newly computed experimental outputs are stored under",OUTPUT)

## 07 — Paired BRDF effect, spatial holdout scatterplots and figure exports

This block reads completed Section 06 heldout files, compares **paired test cells** (same resolution and split), and generates the main per-scale and per-site MAE plots plus scatterplots. It never trains or alters results. Missing scenarios are listed explicitly rather than represented as measured.

In [ ]:
# 07 — Summarize experiments; paired same-cell BRDF-minus-RAW
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
rows=[];paired=[];pred_tables=[]
for scale in SCALES:
    for rep in range(REPEATS):
        data={}
        for variant in ("RAW","BRDF"):
            f=OUTPUT/"04_validation"/f"{scale}m"/variant/f"heldout_rep{rep}.parquet"
            if f.exists():
                p=pd.read_parquet(f)
                p["scale_m"]=scale;p["variant"]=variant;p["repeat"]=rep
                data[variant]=p;pred_tables.append(p)
                for site,sub in p.groupby("site"):
                    y=sub.observed.to_numpy(float);pred=sub.predicted.to_numpy(float)
                    rows.append(dict(scale_m=scale,variant=variant,repeat=rep,site=site,
                        n=len(y),R2=r2_score(y,pred) if np.var(y)>1e-12 else np.nan,
                        MAE_pp=100*np.mean(np.abs(pred-y)),bias_pp=100*np.mean(pred-y),
                        high_bias_pp=100*np.mean((pred-y)[y>=.4]) if (y>=.4).any() else np.nan))
        if set(data)=={"RAW","BRDF"}:
            keys=["site","row","col"]
            m=data["RAW"].merge(data["BRDF"],on=keys,validate="one_to_one",suffixes=("_raw","_brdf"))
            assert len(m)==len(data["RAW"])==len(data["BRDF"]),("Test sets differ",scale,rep)
            assert np.allclose(m.observed_raw,m.observed_brdf,atol=1e-7)
            for site,p in m.groupby("site"):
                raw=100*np.mean(np.abs(p.predicted_raw-p.observed_raw))
                brdf=100*np.mean(np.abs(p.predicted_brdf-p.observed_brdf))
                paired.append(dict(scale_m=scale,repeat=rep,site=site,
                    raw_MAE_pp=raw,brdf_MAE_pp=brdf,
                    change_MAE_pp=brdf-raw))
if not rows:
    print("No validation runs yet: set RUN_MODELS=True after preparing the satellite samples.")
else:
    metrics=pd.DataFrame(rows)
    metrics.to_csv(OUTPUT/"04_validation"/"heldout_metrics_all.csv",index=False)
    report=metrics.groupby(["scale_m","variant","site"]).agg(
        MAE_pp=("MAE_pp","mean"),MAE_sd=("MAE_pp","std"),
        R2=("R2","mean"),bias_pp=("bias_pp","mean"),
        high_bias_pp=("high_bias_pp","mean")).reset_index()
    display(report.round(3))
    report.to_csv(OUTPUT/"04_validation"/"report_site_metrics.csv",index=False)
    fig,ax=plt.subplots(figsize=(9,5))
    for variant,g in report.groupby("variant"):
        means=g.groupby("scale_m").MAE_pp.mean()
        ax.plot(means.index,means.values,marker="o",label=variant)
    ax.set(xlabel="Cell size (m)",ylabel="Spatial holdout MAE (percentage points)",
           title="Equal-forest average across available sites")
    ax.legend();fig.tight_layout()
    fig.savefig(OUTPUT/"05_figures"/"MAE_vs_resolution_raw_BRDF.png",dpi=200)
    plt.show()
    pred=pd.concat(pred_tables,ignore_index=True)
    pred.to_parquet(OUTPUT/"04_validation"/"all_heldout_predictions.parquet",index=False)
if paired:
    pair=pd.DataFrame(paired)
    pair.to_csv(OUTPUT/"04_validation"/"paired_BRDF_effect.csv",index=False)
    display(pair.groupby(["scale_m","site"]).change_MAE_pp.agg(["mean","std"]).round(3))
else:print("Paired BRDF result unavailable until both corrected and raw inputs have passed all stages.")
print("Output directory:",OUTPUT)